# Edge-DP KL-Perturbed Dictionary — Bare-Bones Driver

Minimal end-to-end driver for the edge-DP synthetic-graph training approach on **Cora**.

Pipeline:
1. Load Cora; stratified public/private/val splits with inductive edge isolation.
2. Train a **public GCN encoder** on public nodes only → `x_enc` (used as feature space everywhere downstream).
3. Build the **best-performing public dictionary** (`full` mode from `dict_pruning.ipynb`):
   - Stage A: overcomplete candidate pool (anchor-centric prototype key — anchor's 1-hop neighbor mean).
   - Stage B: hard structural / purity / overlap filters.
   - Stage C: k-center diversification + held-out public-query coverage.
4. **Edge-DP exponential mechanism** assignment with **soft-regularized query normalization** (Strategy 5):
$$T_\tau(q)=\frac{q}{\sqrt{\lVert q\rVert_2^2+\tau}}.$$ Globally Lipschitz, so the neg-L2 sensitivity argument for the one-hop mean stays clean.
5. Train a small GCN on the assigned prototype subgraphs and report basic metrics.

In [ ]:
# ===================================================================
# CELL 1: Imports & global config
# ===================================================================
import os, time, math, random
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    subgraph, k_hop_subgraph, to_undirected, coalesce, dropout_edge,
)
from torch_geometric.nn import GCNConv, global_mean_pool

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Active device: {device}')

CONFIG = {
    'seed': 42,
    # splits
    'public_frac': 0.20,
    'val_frac': 0.20,
    'public_query_frac': 0.25,   # within public, held out for Stage-C coverage scoring
    # mechanism
    'epsilon': 1.0,
    'utility_sensitivity': 1.0,
    'query_mode': 'one_hop',
    'walk_hops': 1,
    'query_hops': 1,
    'label_conditioning': True,
    'tau_soft_norm': 1e-3,       # Strategy 5 regularizer (soft denominator clip)
    # dictionary
    'dict_per_class': 8,
    'pool_mult': 6,              # Stage A overcomplete factor
    'min_class_fraction': 1.0,
    'max_proto_nodes': 32,
    # Stage B filters
    'min_proto_nodes': 2,
    'min_proto_edges': 1,
    'purity_floor': 0.5,
    'require_connected': True,
    'max_overlap_frac': 0.85,
    # Stage C
    'kcenter_lambda': 0.5,
    # public encoder
    'encoder_hidden': 64,
    'encoder_out': 32,
    'encoder_epochs': 200,
    'encoder_lr': 0.01,
    'encoder_wd': 5e-4,
    'encoder_dropout': 0.5,
    # downstream GCN training
    'epochs': 50,
    'batch_size': 32,
    'lr': 0.01,
    'feature_jitter_std': 0.05,
    'edge_dropout_p': 0.10,
}

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

In [ ]:
# ===================================================================
# CELL 2: Load Cora & build splits with inductive edge isolation
# ===================================================================
print('Loading Cora...')
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
data.x = F.normalize(data.x.float(), p=2, dim=1)
data.y = data.y.long()
num_classes = int(dataset.num_classes)
data.edge_index = coalesce(
    to_undirected(data.edge_index, num_nodes=data.num_nodes),
    num_nodes=data.num_nodes,
)
labels = data.y
print(f'Cora: {data.num_nodes} nodes | {data.edge_index.size(1)} edges | {num_classes} classes')


def stratified_split_indices(y, public_frac, val_frac, seed):
    g = torch.Generator().manual_seed(seed)
    pub, val, tr = [], [], []
    for c in torch.unique(y).tolist():
        idx = torch.where(y == c)[0]
        if idx.numel() == 0:
            continue
        perm = idx[torch.randperm(idx.numel(), generator=g)]
        n_pub = max(1, int(public_frac * idx.numel()))
        n_val = max(1, int(val_frac * idx.numel()))
        pub.append(perm[:n_pub])
        val.append(perm[n_pub:n_pub + n_val])
        tr.append(perm[n_pub + n_val:])
    return torch.cat(pub), torch.cat(tr), torch.cat(val)


def split_public_pool_query(public_nodes, labels, query_frac, seed):
    g = torch.Generator().manual_seed(seed)
    pool, query = [], []
    for c in torch.unique(labels[public_nodes]).tolist():
        idx = public_nodes[labels[public_nodes] == c]
        if idx.numel() == 0:
            continue
        perm = idx[torch.randperm(idx.numel(), generator=g)]
        n_q = max(1, int(query_frac * idx.numel()))
        n_q = min(n_q, max(idx.numel() - 1, 0))
        query.append(perm[:n_q])
        pool.append(perm[n_q:])
    return torch.cat(pool), torch.cat(query)


public_nodes, train_nodes, val_nodes = stratified_split_indices(
    labels, CONFIG['public_frac'], CONFIG['val_frac'], seed=CONFIG['seed'])
public_pool_nodes, public_query_nodes = split_public_pool_query(
    public_nodes, labels, CONFIG['public_query_frac'], seed=CONFIG['seed'])

pub_edge_index, _      = subgraph(public_nodes,      data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
priv_edge_index, _     = subgraph(train_nodes,       data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
val_edge_index, _      = subgraph(val_nodes,         data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
pool_edge_index, _     = subgraph(public_pool_nodes, data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)

print(f'public: {public_nodes.numel()} (pool={public_pool_nodes.numel()}, query={public_query_nodes.numel()}) | '
      f'private train: {train_nodes.numel()} | val: {val_nodes.numel()}')

In [ ]:
# ===================================================================
# CELL 3: Public GCN encoder (no DP cost) → x_enc
# Trains a 2-layer GCN node classifier on public nodes/edges only.
# We then apply it to all nodes to extract a 32-d penultimate embedding.
# Same-class nodes cluster more tightly in this space, so L2 distance is a
# more meaningful similarity proxy for both Stage-C selection and the EM.
# ===================================================================
class PublicNodeEncoder(nn.Module):
    def __init__(self, in_channels, hidden, out, num_classes, dropout):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out)
        self.head = nn.Linear(out, num_classes)
        self.dropout = dropout

    def encode(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        return x

    def forward(self, x, edge_index):
        return self.head(self.encode(x, edge_index))


def train_public_encoder(x_features, pub_edges, labels, public_nodes, num_classes,
                          hidden, out, epochs, lr, wd, dropout, seed):
    set_seed(seed)
    x_d   = x_features.to(device)
    e_d   = pub_edges.to(device)
    y_d   = labels.to(device)
    pub_mask = torch.zeros(x_d.size(0), dtype=torch.bool, device=device)
    pub_mask[public_nodes] = True

    model = PublicNodeEncoder(x_d.size(1), hidden, out, num_classes, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(x_d, e_d)[pub_mask], y_d[pub_mask])
        loss.backward(); opt.step()
        if epoch % 50 == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                pred = model(x_d, e_d)[pub_mask].argmax(dim=1)
                acc = float((pred == y_d[pub_mask]).float().mean().item())
            if acc > best_acc:
                best_acc = acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'  encoder epoch {epoch:3d} | loss={loss.item():.4f} | pub_acc={acc:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        x_enc = F.normalize(model.encode(x_d, e_d), p=2, dim=1).cpu()
    print(f'Encoder done. x_enc dim={x_enc.size(1)} | best public-node acc={best_acc:.4f}')
    return x_enc


x_enc = train_public_encoder(
    data.x, pub_edge_index, labels, public_nodes, num_classes,
    hidden=CONFIG['encoder_hidden'], out=CONFIG['encoder_out'],
    epochs=CONFIG['encoder_epochs'], lr=CONFIG['encoder_lr'],
    wd=CONFIG['encoder_wd'], dropout=CONFIG['encoder_dropout'],
    seed=CONFIG['seed'],
)

In [ ]:
# ===================================================================
# CELL 4: Mechanism helpers
#   - one_hop_mean_aggregate
#   - soft_normalize  (Strategy 5: T_tau(q) = q / sqrt(||q||^2 + tau))
#   - build_prototype_feature  (anchor-centric: anchor's 1-hop neighbor mean)
#   - build_private_query_features  (uses soft_normalize, NOT F.normalize)
# ===================================================================
TAU = CONFIG['tau_soft_norm']


def one_hop_mean_aggregate(edge_index, x_features, num_nodes):
    """H[u] = mean_{v in N(u)} x_v on the given (sub)graph edge set."""
    eu = coalesce(to_undirected(edge_index, num_nodes=num_nodes), num_nodes=num_nodes)
    H = torch.zeros_like(x_features)
    if eu.numel() == 0:
        return H
    row, col = eu
    H.scatter_add_(0, row.unsqueeze(1).expand(-1, x_features.size(1)), x_features[col])
    counts = torch.zeros((num_nodes,), dtype=x_features.dtype, device=x_features.device)
    counts.scatter_add_(0, row, torch.ones_like(row, dtype=x_features.dtype))
    out = torch.zeros_like(H)
    nz = counts > 0
    out[nz] = H[nz] / counts[nz].unsqueeze(1)
    return out


def soft_normalize(x, tau=TAU, dim=-1):
    """Soft regularized normalization (Strategy 5):
        T_tau(q) = q / sqrt(||q||^2 + tau)
    Globally bounded (||T_tau(q)|| < 1) and globally Lipschitz, so the neg-L2
    utility on the one-hop mean has bounded sensitivity even near zero.
    Approaches q / ||q||_2 when ||q||_2 >> sqrt(tau), preserving the angular
    matching geometry against L2-normalized prototypes.
    """
    sq = (x * x).sum(dim=dim, keepdim=True)
    return x / torch.sqrt(sq + tau)


def build_prototype_feature(edge_index_sub, x_sub, anchor_local_idx, query_mode='one_hop'):
    """Anchor-centric prototype key (mirrors the per-node private query):
        proto_feat = normalize(H1_nodes[anchor_local_idx])
    where H1_nodes is the 1-hop neighbor mean computed on the prototype subgraph.
    """
    n = x_sub.size(0)
    H1_nodes = one_hop_mean_aggregate(edge_index_sub, x_sub, num_nodes=n)
    z1 = H1_nodes[anchor_local_idx]
    if query_mode == 'one_hop':
        return F.normalize(z1, p=2, dim=0)
    if query_mode == 'two_hop_concat':
        H2_nodes = one_hop_mean_aggregate(edge_index_sub, H1_nodes, num_nodes=n)
        z2 = H2_nodes[anchor_local_idx]
        return F.normalize(torch.cat([z1, z2], dim=0), p=2, dim=0)
    raise ValueError(f'Unsupported query_mode: {query_mode}')


def build_private_query_features(private_edges, x_features, query_mode='one_hop', tau=TAU):
    """Per-private-node query vector with soft regularized normalization.
    Replaces the previous F.normalize(...) (which is not globally Lipschitz at zero).
    """
    n = x_features.size(0)
    H1 = one_hop_mean_aggregate(private_edges, x_features, num_nodes=n)
    if query_mode == 'one_hop':
        return soft_normalize(H1, tau=tau, dim=1)
    if query_mode == 'two_hop_concat':
        H2 = one_hop_mean_aggregate(private_edges, H1, num_nodes=n)
        return soft_normalize(torch.cat([H1, H2], dim=1), tau=tau, dim=1)
    raise ValueError(f'Unsupported query_mode: {query_mode}')


def _is_connected_subgraph(node_ids, edge_index, num_nodes):
    if node_ids.numel() <= 1:
        return True
    sub_edges, _ = subgraph(node_ids, edge_index, relabel_nodes=True, num_nodes=num_nodes)
    if sub_edges.numel() == 0:
        return False
    n = node_ids.numel()
    adj = defaultdict(list)
    for a, b in zip(sub_edges[0].tolist(), sub_edges[1].tolist()):
        adj[a].append(b)
    seen = {0}; stack = [0]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v); stack.append(v)
    return len(seen) == n

In [ ]:
# ===================================================================
# CELL 5: Build the public dictionary (full pipeline = best mode)
#   Stage A — overcomplete candidate pool (anchor-centric proto_feat).
#   Stage B — hard structural / purity / overlap filters.
#   Stage C — k-center diversification + held-out public-query coverage.
# All stages use x_enc. No private edges are touched here.
# ===================================================================

def build_candidate_pool(data_obj, x_features, labels, pool_nodes, pool_edges,
                          num_classes, target_per_class, num_hops, query_mode,
                          max_proto_nodes, min_class_fraction, rng_seed):
    pool_und = coalesce(
        to_undirected(pool_edges, num_nodes=data_obj.num_nodes),
        num_nodes=data_obj.num_nodes,
    )
    g = torch.Generator().manual_seed(int(rng_seed))
    pool_labels = labels[pool_nodes]
    candidates = []
    for c in range(num_classes):
        class_pool = pool_nodes[pool_labels == c]
        if class_pool.numel() == 0:
            continue
        for _ in range(target_per_class):
            pick = torch.randint(0, class_pool.numel(), (1,), generator=g).item()
            anchor = class_pool[pick]
            subset, _, _, _ = k_hop_subgraph(
                int(anchor.item()), num_hops=num_hops,
                edge_index=pool_und, relabel_nodes=False,
                num_nodes=data_obj.num_nodes,
            )
            proto_labels = labels[subset]
            class_mask = proto_labels == c
            class_count = int(class_mask.sum().item())
            subset_count = int(subset.numel())
            frac = class_count / max(subset_count, 1)

            if frac >= min_class_fraction and class_count > 0:
                class_subset = subset
            else:
                class_subset = subset[class_mask]
                if class_subset.numel() == 0:
                    class_subset = anchor.view(1)

            # Anchor must always be in the prototype subgraph (anchor-centric proto_feat).
            if not torch.any(class_subset == anchor):
                class_subset = torch.unique(torch.cat([class_subset, anchor.view(1)]))

            if class_subset.numel() > max_proto_nodes:
                non_anchor = class_subset[class_subset != anchor]
                budget = max_proto_nodes - 1
                if budget <= 0:
                    class_subset = anchor.view(1)
                else:
                    if non_anchor.numel() > budget:
                        perm = torch.randperm(non_anchor.numel(), generator=g)[:budget]
                        non_anchor = non_anchor[perm]
                    class_subset = torch.cat([anchor.view(1), non_anchor])

            sub_edges, _ = subgraph(class_subset, pool_und, relabel_nodes=True,
                                     num_nodes=data_obj.num_nodes)
            x_sub = x_features[class_subset]
            anchor_local_idx = int((class_subset == anchor).nonzero(as_tuple=True)[0].item())
            proto_feat = build_prototype_feature(sub_edges, x_sub, anchor_local_idx, query_mode)

            candidates.append({
                'class': c,
                'anchor': int(anchor.item()),
                'node_ids': class_subset.clone(),
                'x': x_sub,
                'edge_index': sub_edges,
                'proto_feat': proto_feat,
                'n_nodes': int(class_subset.numel()),
                'n_edges': int(sub_edges.size(1)),
                'purity': frac,
            })
    return candidates


def hard_filter_pool(pool, pub_edges, num_nodes, min_nodes, min_edges, purity_floor,
                      require_connected, max_overlap_frac):
    pub_und = coalesce(to_undirected(pub_edges, num_nodes=num_nodes), num_nodes=num_nodes)
    kept_per_class = defaultdict(list); kept = []; reasons = Counter()
    for p in pool:
        if p['n_nodes'] < min_nodes:    reasons['min_nodes'] += 1; continue
        if p['n_edges'] < min_edges:    reasons['min_edges'] += 1; continue
        if p['purity']  < purity_floor: reasons['purity']    += 1; continue
        if require_connected and not _is_connected_subgraph(p['node_ids'], pub_und, num_nodes):
            reasons['disconnected'] += 1; continue
        s = set(p['node_ids'].tolist()); overlap = False
        for prev_set, _ in kept_per_class[p['class']]:
            inter = len(s & prev_set)
            if inter / max(len(s | prev_set), 1) >= max_overlap_frac:
                overlap = True; break
        if overlap:
            reasons['overlap'] += 1; continue
        kept_per_class[p['class']].append((s, p))
        kept.append(p)
    return kept, reasons


@torch.no_grad()
def compute_public_query_features(query_nodes, pool_edges, x_features, labels, num_classes,
                                    query_mode, query_hops, num_total_nodes):
    """Per held-out public-query-node feature in the same (anchor-centric) space as proto_feat."""
    pool_und = coalesce(
        to_undirected(pool_edges, num_nodes=num_total_nodes),
        num_nodes=num_total_nodes,
    )
    per_class = {c: [] for c in range(num_classes)}
    for node in query_nodes.tolist():
        subset, _, _, _ = k_hop_subgraph(
            node, num_hops=query_hops, edge_index=pool_und,
            relabel_nodes=False, num_nodes=num_total_nodes,
        )
        if subset.numel() == 0:
            subset = torch.tensor([node], dtype=torch.long)
        if not torch.any(subset == node):
            subset = torch.unique(torch.cat([subset, torch.tensor([node])]))
        sub_edges, _ = subgraph(subset, pool_und, relabel_nodes=True, num_nodes=num_total_nodes)
        anchor_local_idx = int((subset == node).nonzero(as_tuple=True)[0].item())
        q_feat = build_prototype_feature(sub_edges, x_features[subset], anchor_local_idx, query_mode)
        per_class[int(labels[node].item())].append(q_feat)
    return {c: torch.stack(L, dim=0) if L else torch.empty(0) for c, L in per_class.items()}


@torch.no_grad()
def diversify_and_cover(kept_pool, candidate_pool, query_features_per_class,
                         num_classes, dict_per_class, kcenter_lambda, rng_seed):
    g = torch.Generator().manual_seed(int(rng_seed))
    by_kept = defaultdict(list)
    for p in kept_pool: by_kept[p['class']].append(p)
    by_all = defaultdict(list)
    for p in candidate_pool: by_all[p['class']].append(p)

    selected = []
    class_to_proto = {c: [] for c in range(num_classes)}
    for c in range(num_classes):
        cands = list(by_kept.get(c, []))
        if len(cands) < dict_per_class:
            cand_ids = {id(p) for p in cands}
            spares = [p for p in by_all.get(c, []) if id(p) not in cand_ids]
            random.Random(int(rng_seed) + c).shuffle(spares)
            cands += spares[:max(dict_per_class - len(cands), 0)]
        if len(cands) == 0:
            continue
        feats = torch.stack([p['proto_feat'] for p in cands], dim=0)
        Q = query_features_per_class.get(c, None)
        if Q is None or Q.numel() == 0:
            start_idx = int(torch.randint(0, len(cands), (1,), generator=g).item())
            running_cov = None
        else:
            single_cov = torch.cdist(Q, feats, p=2).mean(dim=0)
            start_idx = int(single_cov.argmin().item())
            running_cov = torch.linalg.norm(Q - feats[start_idx].unsqueeze(0), dim=1)
        chosen = [start_idx]
        dist_to_sel = torch.linalg.norm(feats - feats[start_idx].unsqueeze(0), dim=1)
        budget = min(dict_per_class, len(cands))
        while len(chosen) < budget:
            div_score = dist_to_sel
            if running_cov is not None:
                d_to_cands = torch.cdist(Q, feats, p=2)
                new_cov = torch.minimum(running_cov.unsqueeze(1), d_to_cands).mean(dim=0)
                cov_score = -new_cov
            else:
                cov_score = torch.zeros(len(cands))
            score = kcenter_lambda * div_score + (1.0 - kcenter_lambda) * cov_score
            for ci in chosen: score[ci] = float('-inf')
            nxt = int(score.argmax().item())
            chosen.append(nxt)
            dist_to_sel = torch.minimum(
                dist_to_sel,
                torch.linalg.norm(feats - feats[nxt].unsqueeze(0), dim=1),
            )
            if running_cov is not None:
                running_cov = torch.minimum(
                    running_cov,
                    torch.linalg.norm(Q - feats[nxt].unsqueeze(0), dim=1),
                )
        for li in chosen:
            class_to_proto[c].append(len(selected))
            selected.append(cands[li])
    return selected, class_to_proto


set_seed(CONFIG['seed'])
print('Stage A: building overcomplete candidate pool...')
candidate_pool = build_candidate_pool(
    data_obj=data, x_features=x_enc, labels=labels,
    pool_nodes=public_pool_nodes, pool_edges=pool_edge_index,
    num_classes=num_classes,
    target_per_class=CONFIG['pool_mult'] * CONFIG['dict_per_class'],
    num_hops=CONFIG['walk_hops'], query_mode=CONFIG['query_mode'],
    max_proto_nodes=CONFIG['max_proto_nodes'],
    min_class_fraction=CONFIG['min_class_fraction'],
    rng_seed=CONFIG['seed'],
)
print(f'  pool size = {len(candidate_pool)}')

print('Stage B: hard filter...')
kept_pool, drop_reasons = hard_filter_pool(
    candidate_pool, pub_edge_index, data.num_nodes,
    min_nodes=CONFIG['min_proto_nodes'], min_edges=CONFIG['min_proto_edges'],
    purity_floor=CONFIG['purity_floor'],
    require_connected=CONFIG['require_connected'],
    max_overlap_frac=CONFIG['max_overlap_frac'],
)
print(f'  kept = {len(kept_pool)} | drops = {dict(drop_reasons)}')

print('Stage C: diversify + cover (k-center on x_enc)...')
query_features_per_class = compute_public_query_features(
    public_query_nodes, pool_edge_index, x_enc, labels, num_classes,
    CONFIG['query_mode'], CONFIG['query_hops'], data.num_nodes,
)
public_dict, class_to_proto_indices = diversify_and_cover(
    kept_pool, candidate_pool, query_features_per_class,
    num_classes=num_classes,
    dict_per_class=CONFIG['dict_per_class'],
    kcenter_lambda=CONFIG['kcenter_lambda'],
    rng_seed=CONFIG['seed'],
)
public_proto_feats = torch.stack([p['proto_feat'] for p in public_dict], dim=0)
print(f'  dictionary size = {len(public_dict)} | feature dim = {public_proto_feats.size(1)}')
print(f'  prototypes per class: '
      f'{ {c: len(idxs) for c, idxs in class_to_proto_indices.items()} }')

In [ ]:
# ===================================================================
# CELL 6: Edge-DP exponential mechanism (per-private-node prototype assignment)
#   - Private query: q_u = soft_normalize(H1[u]) on private edges (Strategy 5).
#   - Utility: u(q, k) = -||proto_feat_k - q||_2  (neg-L2).
#   - Edge-level composition: a single private edge affects <=2 node queries,
#     so we split the total epsilon over two releases.
# ===================================================================

def gumbel_max_sample(logits):
    u = torch.rand_like(logits).clamp_(1e-8, 1 - 1e-8)
    return torch.argmax(logits + (-torch.log(-torch.log(u)))).item()


@torch.no_grad()
def synthesize_edge_dp_assignments(private_edges, x_features, labels, private_nodes,
                                     dict_features, class_to_proto_indices,
                                     epsilon_total, utility_sensitivity,
                                     query_mode, label_conditioning, tau):
    eps_per = epsilon_total / 2.0
    Q = build_private_query_features(private_edges, x_features, query_mode=query_mode, tau=tau)
    K = dict_features.size(0)
    nc = int(labels.max().item()) + 1
    all_idx = torch.arange(K, dtype=torch.long)
    class_idx = {
        c: (torch.tensor(class_to_proto_indices.get(c, []), dtype=torch.long)
            if class_to_proto_indices.get(c) else all_idx)
        for c in range(nc)
    }
    N = int(private_nodes.numel())
    sel = torch.empty(N, dtype=torch.long)
    priv_labels = labels[private_nodes].long().clone()
    counts = torch.zeros(K, dtype=torch.long)
    ent_sum = top_sum = 0.0
    for j, u in enumerate(private_nodes.tolist()):
        y_u = int(labels[u].item())
        cand_idx = class_idx[y_u] if label_conditioning else all_idx
        cand_feats = dict_features.index_select(0, cand_idx)
        d = torch.linalg.norm(cand_feats - Q[u].unsqueeze(0), dim=1)
        logits = (eps_per / (2.0 * utility_sensitivity)) * (-d)
        probs = torch.softmax(logits, dim=0)
        ent_sum += float(-(probs * torch.log(probs.clamp_min(1e-12))).sum().item())
        top_sum += float(probs.max().item())
        li = gumbel_max_sample(logits)
        pi = int(cand_idx[li].item())
        sel[j] = pi
        counts[pi] += 1
    diag = {
        'epsilon_total': float(epsilon_total),
        'epsilon_per_node': float(eps_per),
        'mean_entropy': ent_sum / max(N, 1),
        'mean_top_probability': top_sum / max(N, 1),
        'unique_selected_ratio': float((counts > 0).sum().item()) / float(max(K, 1)),
    }
    return {'proto_indices': sel, 'labels': priv_labels}, diag


set_seed(CONFIG['seed'])
print(f'Edge-DP exponential mechanism (epsilon_total={CONFIG["epsilon"]:.3f}, tau={TAU})')
assignments, syn_diag = synthesize_edge_dp_assignments(
    priv_edge_index, x_enc, labels, train_nodes,
    public_proto_feats, class_to_proto_indices,
    epsilon_total=CONFIG['epsilon'],
    utility_sensitivity=CONFIG['utility_sensitivity'],
    query_mode=CONFIG['query_mode'],
    label_conditioning=CONFIG['label_conditioning'],
    tau=CONFIG['tau_soft_norm'],
)
print(f'  epsilon_per_node      = {syn_diag["epsilon_per_node"]:.4f}')
print(f'  mean entropy          = {syn_diag["mean_entropy"]:.4f}')
print(f'  mean top-1 prob       = {syn_diag["mean_top_probability"]:.4f}')
print(f'  unique-prototype-ratio= {syn_diag["unique_selected_ratio"]:.2%}')

In [ ]:
# ===================================================================
# CELL 7: Train a small GCN on the assigned prototype subgraphs
# (post-processing of DP-safe assignments, no extra privacy cost).
# ===================================================================
class PrototypeAssignmentDataset(torch.utils.data.Dataset):
    def __init__(self, public_dict, proto_indices, labels):
        self.public_dict = public_dict
        self.proto_indices = proto_indices.long().cpu()
        self.labels = labels.long().cpu()
    def __len__(self):
        return self.proto_indices.numel()
    def __getitem__(self, idx):
        p = self.public_dict[int(self.proto_indices[idx].item())]
        return Data(x=p['x'], edge_index=p['edge_index'], y=self.labels[idx].view(1))


class StandardGCN(nn.Module):
    def __init__(self, hidden, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = nn.Linear(hidden, num_classes)
    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)


def build_validation_loader(x_features, labels, val_nodes, val_edges, query_hops, num_total_nodes):
    vu = coalesce(to_undirected(val_edges, num_nodes=num_total_nodes), num_nodes=num_total_nodes)
    dl = []
    for n in val_nodes.tolist():
        subset, sei, _, _ = k_hop_subgraph(
            n, num_hops=query_hops, edge_index=vu,
            relabel_nodes=True, num_nodes=num_total_nodes,
        )
        dl.append(Data(x=x_features[subset], edge_index=sei, y=labels[n].unsqueeze(0)))
    return DataLoader(dl, batch_size=32, shuffle=False)


def train_gnn(assignments, public_dict, val_loader, num_features, num_classes,
                epochs, batch_size, lr, feature_jitter_std, edge_dropout_p):
    ds = PrototypeAssignmentDataset(public_dict, assignments['proto_indices'], assignments['labels'])
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = StandardGCN(32, num_features, num_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    best = 0.0
    for epoch in range(1, epochs + 1):
        model.train(); total_loss = 0.0; train_correct = 0
        for batch in dl:
            batch = batch.to(device)
            opt.zero_grad(set_to_none=True)
            xa = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            ea, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if ea.numel() == 0:
                ea = batch.edge_index
            out = model(xa, ea, batch.batch)
            loss = crit(out, batch.y)
            loss.backward(); opt.step()
            total_loss += loss.item() * batch.num_graphs
            train_correct += int((out.argmax(dim=1) == batch.y).sum())
        train_acc = train_correct / len(dl.dataset)

        model.eval(); val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())
        val_acc = val_correct / len(val_loader.dataset)
        best = max(best, val_acc)
        if epoch % 10 == 0 or epoch == 1:
            print(f'  epoch {epoch:03d} | train_loss={total_loss/len(dl.dataset):.4f} | '
                  f'train_acc={train_acc:.4f} | val_acc={val_acc:.4f}')
    return best


val_loader = build_validation_loader(
    x_enc, labels, val_nodes, val_edge_index, CONFIG['query_hops'], data.num_nodes,
)
print(f'Validation set: {len(val_loader.dataset)} graphs')

# Majority-class baseline
val_counts = torch.bincount(labels[val_nodes], minlength=num_classes)
majority_acc = float((labels[val_nodes] == int(val_counts.argmax().item())).float().mean().item())
print(f'Majority-class baseline (val): {majority_acc:.4f}')

print('\n--- Training GCN on edge-DP synthetic assignments ---')
set_seed(CONFIG['seed'])
best_val = train_gnn(
    assignments, public_dict, val_loader,
    num_features=x_enc.size(1), num_classes=num_classes,
    epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'], lr=CONFIG['lr'],
    feature_jitter_std=CONFIG['feature_jitter_std'],
    edge_dropout_p=CONFIG['edge_dropout_p'],
)

print('\n=== Summary ===')
print(f'Dataset                : Cora')
print(f'Epsilon (total)        : {CONFIG["epsilon"]}')
print(f'Tau (soft-norm)        : {CONFIG["tau_soft_norm"]}')
print(f'Dictionary size        : {len(public_dict)}')
print(f'Mean selection entropy : {syn_diag["mean_entropy"]:.4f}')
print(f'Mean top-1 probability : {syn_diag["mean_top_probability"]:.4f}')
print(f'Majority baseline acc  : {majority_acc:.4f}')
print(f'Best validation acc    : {best_val:.4f}')